In [ ]:
import os
import numpy as np
import tensorflow as tf

from tensorflow.keras.preprocessing.image import load_img, img_to_array
from sklearn.model_selection import train_test_split


# --------------------------------------------------
# 1. Set the path to the directory containing images
# --------------------------------------------------
faces_dir = "D:/R2021 DL LAB/Faces/Faces"


# --------------------------------------------------
# 2. Load face images and labels
# --------------------------------------------------
x_data = []
y_data = []

for filename in os.listdir(faces_dir):

    # Process only JPG images
    if filename.lower().endswith(".jpg"):

        img_path = os.path.join(faces_dir, filename)

        # Resize image to 64 x 64
        img = load_img(img_path, target_size=(64, 64))

        # Convert image to NumPy array
        img_array = img_to_array(img)

        x_data.append(img_array)

        # Example filename:
        # person_0.jpg
        # person_1.jpg
        #
        # Extract the label after "_"
        label = filename.split("_")[1].split(".")[0]

        y_data.append(label)


# --------------------------------------------------
# 3. Convert data to NumPy arrays
# --------------------------------------------------
x_data = np.array(x_data)
y_data = np.array(y_data)


# --------------------------------------------------
# 4. Convert labels to integers
# --------------------------------------------------
y_data_numeric = np.array([int(label) for label in y_data])


# --------------------------------------------------
# 5. Convert labels to one-hot encoded vectors
# --------------------------------------------------
num_classes = len(np.unique(y_data_numeric))

y_data_encoded = tf.keras.utils.to_categorical(
    y_data_numeric,
    num_classes=num_classes
)


# --------------------------------------------------
# 6. Normalize image pixel values
# --------------------------------------------------
x_data = x_data / 255.0


# --------------------------------------------------
# 7. Split data into training and validation sets
# --------------------------------------------------
x_train, x_val, y_train, y_val = train_test_split(
    x_data,
    y_data_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_data_numeric
)


# --------------------------------------------------
# 8. Create CNN model
# --------------------------------------------------
model = tf.keras.models.Sequential([
    
    tf.keras.layers.Conv2D(
        32,
        (3, 3),
        activation="relu",
        input_shape=(64, 64, 3)
    ),

    tf.keras.layers.MaxPooling2D(
        (2, 2)
    ),

    tf.keras.layers.Conv2D(
        64,
        (3, 3),
        activation="relu"
    ),

    tf.keras.layers.MaxPooling2D(
        (2, 2)
    ),

    tf.keras.layers.Flatten(),

    tf.keras.layers.Dense(
        128,
        activation="relu"
    ),

    tf.keras.layers.Dense(
        num_classes,
        activation="softmax"
    )
])


# --------------------------------------------------
# 9. Compile the model
# --------------------------------------------------
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)


# --------------------------------------------------
# 10. Train the CNN
# --------------------------------------------------
model.fit(
    x_train,
    y_train,
    epochs=10,
    batch_size=32,
    validation_data=(x_val, y_val)
)


# --------------------------------------------------
# 11. Save the trained model
# --------------------------------------------------
model.save("face_recognition_model.keras")

print("Model trained and saved successfully!")